<a href="https://colab.research.google.com/github/gowrisankar393/vaylen-transitlk/blob/Passenger-Incident-Detction/humandetectionmodelcommit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import yaml
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from sklearn.metrics import roc_curve, auc

# -----------------------------
# CONFIGURATION
# -----------------------------

DATASET_PATH = "dataset"
TRAIN_IMAGES = os.path.join(DATASET_PATH, "images/train")
VAL_IMAGES = os.path.join(DATASET_PATH, "images/val")

DATASET_YAML = "dataset.yaml"

EPOCHS = 100
BATCH_SIZE = 4        # reduced for CPU
IMG_SIZE = 416        # reduced for CPU
SAVE_PERIOD = 10

PROJECT_NAME = "fight_detection"
RUN_NAME = "human_detector"

# -----------------------------
# CREATE DATASET YAML
# -----------------------------

dataset_config = {
    "path": DATASET_PATH,
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "person"
    }
}

with open(DATASET_YAML, "w") as f:
    yaml.dump(dataset_config, f)

print("Dataset YAML created")

# -----------------------------
# LOAD LIGHTWEIGHT MODEL
# -----------------------------

print("Loading YOLOv8n model...")

model = YOLO("yolov8n.pt")

# -----------------------------
# TRAIN MODEL
# -----------------------------

print("Starting training...")

results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device="cpu",
    amp=False,
    workers=2,
    save=True,
    save_period=SAVE_PERIOD,
    project=PROJECT_NAME,
    name=RUN_NAME,
    pretrained=True
)

print("Training completed")

# -----------------------------
# EVALUATE MODEL
# -----------------------------

print("Evaluating model...")



In [ ]:
metrics = model.val()

precision = metrics.box.mp
recall = metrics.box.mr
map50 = metrics.box.map50
map5095 = metrics.box.map

print("\nEvaluation Metrics")
print("------------------")
print("Precision:", precision)
print("Recall:", recall)
print("mAP@0.5:", map50)
print("mAP@0.5:0.95:", map5095)

# -----------------------------
# ROC CURVE GENERATION
# -----------------------------

print("Generating ROC curve...")

y_true = []
y_scores = []

val_images = [os.path.join(VAL_IMAGES, img) for img in os.listdir(VAL_IMAGES)]

for img_path in val_images:

    results = model.predict(img_path, conf=0.001, verbose=False)

    for r in results:

        if r.boxes is None:
            continue

        for box in r.boxes:

            score = float(box.conf)

            y_scores.append(score)
            y_true.append(1)

y_true = np.array(y_true)
y_scores = np.array(y_scores)

if len(y_true) > 0:

    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)

    plt.figure()

    plt.plot(fpr, tpr, label="ROC curve (AUC = %0.2f)" % roc_auc)
    plt.plot([0, 1], [0, 1], linestyle="--")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()

    plt.savefig("roc_curve.png")
    plt.show()

    print("ROC AUC:", roc_auc)

else:

    print("No detections found for ROC calculation")

# -----------------------------
# BEST MODEL LOCATION
# -----------------------------

BEST_MODEL_PATH = f"runs/detect/{PROJECT_NAME}/{RUN_NAME}/weights/best.pt"

print("\nBest model saved at:")
print(BEST_MODEL_PATH)

# -----------------------------
# OPTIONAL REALTIME TEST
# -----------------------------

def realtime_test(video_source=0):

    import cv2

    model = YOLO(BEST_MODEL_PATH)

    cap = cv2.VideoCapture(video_source)

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        results = model(frame)

        annotated = results[0].plot()

        cv2.imshow("Human Detection", annotated)

        if cv2.waitKey(1) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()


# Uncomment to test webcam
# realtime_test(0)